# TP 8 - Calculateur d'itinéraires OSMnx (optionnel)

Ce notebook constitue la seconde partie du TP 8 : charger le réseau routier de
Lyon depuis OpenStreetMap, inspecter ses attributs, trouver les sommets les plus
proches de deux adresses, calculer un itinéraire avec Dijkstra et le tracer sur
une carte.

Il **n'est pas exécuté** dans le dépôt car il nécessite une connexion Internet
et les paquets optionnels `osmnx` / `geopandas`. Installez-les avec :

```bash
pip install -e ".[routing]"
```

In [ ]:
# Décommenter dans un environnement neuf :
# %pip install osmnx geopandas folium

In [ ]:
import networkx as nx
import osmnx as ox

# Tracer les figures à l'intérieur du notebook.
%matplotlib inline

## 1. Charger et afficher le réseau de Lyon

`graph_from_place` télécharge le réseau praticable. Le résultat est un
`MultiDiGraph` : les deux sens sont des arcs distincts, et des arcs parallèles
sont possibles.

In [ ]:
G = ox.graph_from_place("Lyon, France", network_type="drive")
print(type(G))
print("sommets :", G.number_of_nodes())
print("arcs :", G.number_of_edges())

In [ ]:
fig, ax = ox.plot_graph(
    G,
    node_size=0,
    edge_color="yellow",
    edge_linewidth=0.5,
    bgcolor="black",
    figsize=(10, 10),
)

## 2. Inspecter les attributs des arêtes

OSMnx stocke un dictionnaire sur chaque arc : longueur, vitesse, type de route
(highway), nom, géométrie, etc.

In [ ]:
first_edge = next(iter(G.edges(keys=True, data=True)))
print(first_edge)

u, v = first_edge[0], first_edge[1]
print(G.get_edge_data(u, v))

Les arcs peuvent être convertis en un `GeoDataFrame`, qui fournit des cartes
interactives et des opérations spatiales.

In [ ]:
nodes, edges = ox.graph_to_gdfs(G)
print(edges.columns)
edges[["length", "highway", "name"]].head()

## 3. Renuméroter les sommets

Les identifiants OpenStreetMap sont grands et épars. Pour la version de Dijkstra
basée sur des tableaux, nous renumérotons les sommets `0, 1, 2, ...` à l'aide
d'une correspondance explicite.

In [ ]:
mapping = {node: index for index, node in enumerate(G.nodes)}
H = nx.relabel_nodes(G, mapping)
print("premiers identifiants :", list(G.nodes)[:5])
print("nouveaux identifiants :", list(H.nodes)[:5])

## 4. Trouver les sommets les plus proches des deux adresses

`ox.geocode` renvoie `(latitude, longitude)`. `nearest_nodes` attend
`X=longitude`, `Y=latitude`.

In [ ]:
origin_address = "6 Rue Plasson et Chaize, Lyon, France"
destination_address = "21 Rue Thenard, Lyon, France"

origin_lat, origin_lon = ox.geocode(origin_address)
destination_lat, destination_lon = ox.geocode(destination_address)
print(origin_lat, origin_lon)
print(destination_lat, destination_lon)

origin = ox.distance.nearest_nodes(H, X=origin_lon, Y=origin_lat)
destination = ox.distance.nearest_nodes(H, X=destination_lon, Y=destination_lat)
print("sommet d'origine :", origin)
print("sommet de destination :", destination)

## 5. Calculer et tracer l'itinéraire

La fonction `dijkstra_heap` du dépôt gère les dictionnaires imbriqués
`{0: {"length": ...}}` d'un `MultiDiGraph`.

In [ ]:
import time
from sda.tp08_networkx import dijkstra1, dijkstra2, dijkstra_heap

for algorithm in (dijkstra1, dijkstra2, dijkstra_heap):
    start = time.perf_counter()
    distance, route = algorithm(H, origin, destination)
    elapsed = time.perf_counter() - start
    print(f"{algorithm.__name__}: {distance:.1f} m en {elapsed:.2f} s")

print("itinéraire :", route)

In [ ]:
ox.plot_graph_route(
    H,
    route,
    route_color="red",
    route_linewidth=3,
    node_size=0,
    bgcolor="black",
    figsize=(10, 10),
)

## 6. Ce qu'il faut retenir

- OSMnx convertit les données OpenStreetMap en un graphe NetworkX ; chaque
  attribut géographique reste disponible sur les sommets et les arcs.
- Le graphe routier est orienté et comporte des arêtes multiples, le poids d'une
  arête est donc `data[0]["length"]`, et non `data["length"]`.
- La même implémentation de Dijkstra fonctionne sur le graphe synthétique et sur
  le réseau routier réel.
- La renumérotation des sommets rend pratiques les versions basées sur des
  tableaux ; la version avec tas n'en a pas besoin mais conserve une comparaison
  équitable.

Le notebook est volontairement laissé non exécuté dans le dépôt : la sortie
dépend des données OpenStreetMap au moment du téléchargement.